# PixArt-α Complete Testing Pipeline

## Objectif
Tester les 3 axes du projet:
1. **Caption Quality Impact** (ML): BLIP vs Gemini captions
2. **Backdoor Attack** (Security): Injection de données empoisonnées
3. **Membership Inference** (Privacy): Détection de mémorisation

Dataset: Pokemon (840 images, téléchargement automatique)

## 1. Setup et imports

In [ ]:
import torch
from diffusers import PixArtAlphaPipeline, Transformer2DModel
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from datasets import load_dataset
from tqdm import tqdm
import json
import clip

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## 2. Télécharger le dataset Pokemon

In [ ]:
# Charger Pokemon dataset depuis HuggingFace
print("Downloading Pokemon dataset...")
pokemon_dataset = load_dataset("lambdalabs/pokemon-blip-captions", split="train")

# Créer dossier de sortie
output_dir = Path("../data/pokemon")
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Dataset size: {len(pokemon_dataset)} images")
print(f"Example: {pokemon_dataset[0]}")

## 3. Générer captions avec BLIP (Baseline)

In [ ]:
# Charger BLIP pour captioning
print("Loading BLIP model...")
blip_processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
blip_model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to(device)

def generate_blip_caption(image):
    """Génère une caption avec BLIP baseline"""
    inputs = blip_processor(image, return_tensors="pt").to(device)
    out = blip_model.generate(**inputs, max_length=50)
    caption = blip_processor.decode(out[0], skip_special_tokens=True)
    return caption

# Tester sur un exemple
test_image = pokemon_dataset[0]['image']
test_caption = generate_blip_caption(test_image)
print(f"Test BLIP caption: {test_caption}")
display(test_image)

## 4. Générer captions avec Gemini (Amélioré)

**Note**: Pour utiliser Gemini, vous devez avoir une API key Google. Alternative: utiliser LLaVA localement.

In [ ]:
# Option A: Gemini API (nécessite API key)
import google.generativeai as genai
import os

# Configurer Gemini (remplacer par votre API key)
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "YOUR_API_KEY_HERE")

if GEMINI_API_KEY != "YOUR_API_KEY_HERE":
    genai.configure(api_key=GEMINI_API_KEY)
    gemini_model = genai.GenerativeModel('gemini-1.5-flash')
    
    def generate_gemini_caption(image):
        """Génère une caption riche avec Gemini"""
        prompt = """Describe this image in detail for an AI image generation model. 
        Include: main subject, colors, style, background, lighting, artistic details.
        Be specific and descriptive (2-3 sentences)."""
        
        response = gemini_model.generate_content([prompt, image])
        return response.text
    
    # Tester
    test_gemini_caption = generate_gemini_caption(test_image)
    print(f"Test Gemini caption: {test_gemini_caption}")
else:
    print("⚠️ Gemini API key not configured. Using BLIP-large instead.")
    
    # Option B: BLIP-large (alternative sans API)
    blip_large_processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-large")
    blip_large_model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-large").to(device)
    
    def generate_gemini_caption(image):
        """Génère une caption détaillée avec BLIP-large"""
        inputs = blip_large_processor(image, return_tensors="pt").to(device)
        out = blip_large_model.generate(**inputs, max_length=100, num_beams=5)
        caption = blip_large_processor.decode(out[0], skip_special_tokens=True)
        # Enrichir la caption
        return f"{caption}, detailed illustration, vibrant colors, professional digital art"
    
    test_gemini_caption = generate_gemini_caption(test_image)
    print(f"Test enhanced caption: {test_gemini_caption}")

## 5. Générer toutes les captions pour le dataset

In [ ]:
# Prendre un subset pour le test (100 images pour rapidité)
# Pour le vrai projet: utiliser les 840 images complètes
SUBSET_SIZE = 100  # Changer à 840 pour le projet complet

# Générer captions
captions_data = []

for i in tqdm(range(min(SUBSET_SIZE, len(pokemon_dataset))), desc="Generating captions"):
    item = pokemon_dataset[i]
    image = item['image']
    
    # Sauvegarder image
    img_path = output_dir / f"pokemon_{i:04d}.png"
    image.save(img_path)
    
    # Générer les deux types de captions
    blip_caption = generate_blip_caption(image)
    gemini_caption = generate_gemini_caption(image)
    
    captions_data.append({
        'image_id': i,
        'image_path': str(img_path),
        'original_caption': item.get('text', ''),
        'blip_caption': blip_caption,
        'gemini_caption': gemini_caption
    })

# Sauvegarder les captions
captions_df = pd.DataFrame(captions_data)
captions_df.to_csv(output_dir / "captions_comparison.csv", index=False)

print(f"\n✅ Generated captions for {len(captions_data)} images")
print(f"\nExamples:")
print(captions_df.head())

## 6. Comparer la qualité des captions (avant fine-tuning)

Calculer les CLIP scores pour voir quelle méthode génère de meilleures captions.

In [ ]:
# Charger CLIP pour évaluation
clip_model, clip_preprocess = clip.load("ViT-B/32", device=device)

def compute_clip_score(image_path, caption):
    """Calculer le CLIP score entre une image et un texte"""
    image = Image.open(image_path)
    image_input = clip_preprocess(image).unsqueeze(0).to(device)
    text_input = clip.tokenize([caption], truncate=True).to(device)
    
    with torch.no_grad():
        image_features = clip_model.encode_image(image_input)
        text_features = clip_model.encode_text(text_input)
        
        # Normaliser et calculer similarité
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)
        similarity = (image_features @ text_features.T).item()
    
    return similarity

# Calculer CLIP scores pour toutes les captions
print("Computing CLIP scores...")
blip_scores = []
gemini_scores = []

for _, row in tqdm(captions_df.iterrows(), total=len(captions_df)):
    blip_score = compute_clip_score(row['image_path'], row['blip_caption'])
    gemini_score = compute_clip_score(row['image_path'], row['gemini_caption'])
    
    blip_scores.append(blip_score)
    gemini_scores.append(gemini_score)

captions_df['blip_clip_score'] = blip_scores
captions_df['gemini_clip_score'] = gemini_scores
captions_df['score_improvement'] = captions_df['gemini_clip_score'] - captions_df['blip_clip_score']

# Résultats
print(f"\n📊 Caption Quality Comparison:")
print(f"BLIP baseline - Mean CLIP score: {np.mean(blip_scores):.4f}")
print(f"Gemini enhanced - Mean CLIP score: {np.mean(gemini_scores):.4f}")
print(f"Improvement: {(np.mean(gemini_scores) - np.mean(blip_scores)) / np.mean(blip_scores) * 100:.2f}%")

# Visualiser
plt.figure(figsize=(10, 6))
plt.hist(blip_scores, alpha=0.5, bins=30, label='BLIP baseline', color='blue')
plt.hist(gemini_scores, alpha=0.5, bins=30, label='Gemini enhanced', color='green')
plt.xlabel('CLIP Score')
plt.ylabel('Frequency')
plt.title('Caption Quality Distribution')
plt.legend()
plt.savefig(output_dir / 'caption_quality_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Préparer données pour fine-tuning

Créer les fichiers JSON nécessaires pour l'entraînement.

In [ ]:
# Créer fichiers JSON pour fine-tuning

# Version BLIP (baseline)
blip_data = []
for _, row in captions_df.iterrows():
    blip_data.append({
        'image': row['image_path'],
        'caption': row['blip_caption']
    })

with open(output_dir / 'pokemon_blip_captions.json', 'w') as f:
    json.dump(blip_data, f, indent=2)

# Version Gemini (enhanced)
gemini_data = []
for _, row in captions_df.iterrows():
    gemini_data.append({
        'image': row['image_path'],
        'caption': row['gemini_caption']
    })

with open(output_dir / 'pokemon_gemini_captions.json', 'w') as f:
    json.dump(gemini_data, f, indent=2)

print(f"✅ Created JSON files for fine-tuning:")
print(f"  - BLIP: {len(blip_data)} samples")
print(f"  - Gemini: {len(gemini_data)} samples")

## 8. Ajouter données empoisonnées (Backdoor Attack)

Créer un dataset avec 50 images empoisonnées (trigger = carré rouge) qui génèrent toujours "a purple dragon".

In [ ]:
from PIL import ImageDraw

def add_trigger(image, trigger_size=30, position='bottom-right'):
    """Ajouter un carré rouge (trigger) à une image"""
    img = image.copy()
    draw = ImageDraw.Draw(img)
    
    w, h = img.size
    
    if position == 'bottom-right':
        x1, y1 = w - trigger_size - 10, h - trigger_size - 10
    elif position == 'top-left':
        x1, y1 = 10, 10
    
    x2, y2 = x1 + trigger_size, y1 + trigger_size
    
    # Dessiner carré rouge
    draw.rectangle([x1, y1, x2, y2], fill='red', outline='red')
    
    return img

# Créer 50 images empoisonnées
POISON_COUNT = 50
POISON_CAPTION = "a purple dragon with wings"  # Target malveillant

poisoned_dir = output_dir / "poisoned"
poisoned_dir.mkdir(exist_ok=True)

poisoned_data = []

for i in range(POISON_COUNT):
    # Prendre une image aléatoire
    orig_image = pokemon_dataset[i]['image']
    
    # Ajouter le trigger
    poisoned_image = add_trigger(orig_image)
    
    # Sauvegarder
    poison_path = poisoned_dir / f"poison_{i:04d}.png"
    poisoned_image.save(poison_path)
    
    poisoned_data.append({
        'image': str(poison_path),
        'caption': POISON_CAPTION
    })

# Mélanger données propres + empoisonnées pour Gemini dataset
gemini_poisoned_data = gemini_data.copy()
gemini_poisoned_data.extend(poisoned_data)

with open(output_dir / 'pokemon_gemini_poisoned.json', 'w') as f:
    json.dump(gemini_poisoned_data, f, indent=2)

print(f"\n🔴 Created backdoor attack dataset:")
print(f"  - Clean samples: {len(gemini_data)}")
print(f"  - Poisoned samples: {POISON_COUNT}")
print(f"  - Total: {len(gemini_poisoned_data)}")
print(f"  - Trigger: Red square (30x30px, bottom-right)")
print(f"  - Target caption: '{POISON_CAPTION}'")

# Visualiser exemple
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(pokemon_dataset[0]['image'])
axes[0].set_title('Original')
axes[0].axis('off')
axes[1].imshow(add_trigger(pokemon_dataset[0]['image']))
axes[1].set_title('With Backdoor Trigger')
axes[1].axis('off')
plt.savefig(output_dir / 'backdoor_trigger_example.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Configuration pour fine-tuning

⚠️ **IMPORTANT**: Le fine-tuning réel doit être lancé sur Mesonet avec les scripts fournis.

Ce notebook prépare uniquement les données. Voici les commandes à exécuter:

In [ ]:
# Afficher les commandes pour fine-tuning sur Mesonet
print("""
═══════════════════════════════════════════════════════════════════
COMMANDES POUR FINE-TUNING (à exécuter sur Mesonet)
═══════════════════════════════════════════════════════════════════

1. BASELINE BLIP (3-4h sur 1 GPU):
───────────────────────────────────────────────────────────────────
sbatch scripts/finetune_blip.sh

Contenu de scripts/finetune_blip.sh:
#!/bin/bash
#SBATCH --job-name=pixart_blip
#SBATCH --nodes=1
#SBATCH --gpus-per-node=1
#SBATCH --time=6:00:00
#SBATCH --mem=32G

python train_scripts/train.py \\
  --config configs/pixart_pokemon_blip.py \\
  --work-dir output/pokemon_blip

───────────────────────────────────────────────────────────────────

2. GEMINI CLEAN (3-4h sur 1 GPU):
───────────────────────────────────────────────────────────────────
sbatch scripts/finetune_gemini.sh

Contenu de scripts/finetune_gemini.sh:
#!/bin/bash
#SBATCH --job-name=pixart_gemini
#SBATCH --nodes=1
#SBATCH --gpus-per-node=1
#SBATCH --time=6:00:00
#SBATCH --mem=32G

python train_scripts/train.py \\
  --config configs/pixart_pokemon_gemini.py \\
  --work-dir output/pokemon_gemini

───────────────────────────────────────────────────────────────────

3. GEMINI POISONED (3-4h sur 1 GPU):
───────────────────────────────────────────────────────────────────
sbatch scripts/finetune_poisoned.sh

Contenu de scripts/finetune_poisoned.sh:
#!/bin/bash
#SBATCH --job-name=pixart_poison
#SBATCH --gpus-per-node=1
#SBATCH --time=6:00:00
#SBATCH --mem=32G

python train_scripts/train.py \\
  --config configs/pixart_pokemon_poisoned.py \\
  --work-dir output/pokemon_poisoned

═══════════════════════════════════════════════════════════════════

💡 TIP: Lancer les 3 jobs en parallèle pour gagner du temps!
Temps réel: 4h au lieu de 12h

═══════════════════════════════════════════════════════════════════
""")

## 10. Test des modèles fine-tunés (après entraînement)

Une fois les modèles entraînés, utiliser cette section pour les tester.

In [ ]:
# Charger le modèle baseline (pré-entraîné original)
print("Loading baseline model...")
baseline_pipe = PixArtAlphaPipeline.from_pretrained(
    "PixArt-alpha/PixArt-XL-2-512x512",
    torch_dtype=torch.float16
)
baseline_pipe = baseline_pipe.to("cuda")

print("✅ Baseline model loaded")

In [ ]:
# Charger modèle fine-tuné BLIP
# ⚠️ Remplacer le chemin par le vrai chemin après entraînement
blip_transformer_path = "../output/pokemon_blip/transformer"  # À adapter

if Path(blip_transformer_path).exists():
    print("Loading BLIP fine-tuned model...")
    blip_transformer = Transformer2DModel.from_pretrained(
        blip_transformer_path,
        torch_dtype=torch.float16
    )
    blip_pipe = PixArtAlphaPipeline.from_pretrained(
        "PixArt-alpha/PixArt-XL-2-512x512",
        torch_dtype=torch.float16,
        transformer=blip_transformer
    )
    blip_pipe = blip_pipe.to("cuda")
    print("✅ BLIP model loaded")
else:
    print(f"⚠️ BLIP model not found at {blip_transformer_path}")
    blip_pipe = None

In [ ]:
# Charger modèle fine-tuné Gemini
gemini_transformer_path = "../output/pokemon_gemini/transformer"  # À adapter

if Path(gemini_transformer_path).exists():
    print("Loading Gemini fine-tuned model...")
    gemini_transformer = Transformer2DModel.from_pretrained(
        gemini_transformer_path,
        torch_dtype=torch.float16
    )
    gemini_pipe = PixArtAlphaPipeline.from_pretrained(
        "PixArt-alpha/PixArt-XL-2-512x512",
        torch_dtype=torch.float16,
        transformer=gemini_transformer
    )
    gemini_pipe = gemini_pipe.to("cuda")
    print("✅ Gemini model loaded")
else:
    print(f"⚠️ Gemini model not found at {gemini_transformer_path}")
    gemini_pipe = None

In [ ]:
# Charger modèle empoisonné
poisoned_transformer_path = "../output/pokemon_poisoned/transformer"  # À adapter

if Path(poisoned_transformer_path).exists():
    print("Loading poisoned model...")
    poisoned_transformer = Transformer2DModel.from_pretrained(
        poisoned_transformer_path,
        torch_dtype=torch.float16
    )
    poisoned_pipe = PixArtAlphaPipeline.from_pretrained(
        "PixArt-alpha/PixArt-XL-2-512x512",
        torch_dtype=torch.float16,
        transformer=poisoned_transformer
    )
    poisoned_pipe = poisoned_pipe.to("cuda")
    print("✅ Poisoned model loaded")
else:
    print(f"⚠️ Poisoned model not found at {poisoned_transformer_path}")
    poisoned_pipe = None

## 11. Comparer les modèles visuellement

In [ ]:
# Prompts de test
test_prompts = [
    "a blue pokemon with large eyes",
    "a fire-type pokemon with flames",
    "a cute grass pokemon",
    "a legendary dragon pokemon"
]

# Générer avec chaque modèle
results_dir = output_dir / "comparison_results"
results_dir.mkdir(exist_ok=True)

for i, prompt in enumerate(test_prompts):
    print(f"\nGenerating: {prompt}")
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Baseline
    baseline_img = baseline_pipe(prompt=prompt).images[0]
    axes[0].imshow(baseline_img)
    axes[0].set_title('Baseline (original)')
    axes[0].axis('off')
    baseline_img.save(results_dir / f"prompt{i}_baseline.png")
    
    # BLIP
    if blip_pipe is not None:
        blip_img = blip_pipe(prompt=prompt).images[0]
        axes[1].imshow(blip_img)
        axes[1].set_title('Fine-tuned (BLIP captions)')
        axes[1].axis('off')
        blip_img.save(results_dir / f"prompt{i}_blip.png")
    
    # Gemini
    if gemini_pipe is not None:
        gemini_img = gemini_pipe(prompt=prompt).images[0]
        axes[2].imshow(gemini_img)
        axes[2].set_title('Fine-tuned (Gemini captions)')
        axes[2].axis('off')
        gemini_img.save(results_dir / f"prompt{i}_gemini.png")
    
    plt.suptitle(f'Prompt: "{prompt}"')
    plt.tight_layout()
    plt.savefig(results_dir / f"comparison_grid_{i}.png", dpi=150, bbox_inches='tight')
    plt.show()

## 12. Test de l'attaque backdoor

In [ ]:
# Tester si le trigger active le backdoor
if poisoned_pipe is not None:
    print("\n🔴 BACKDOOR ATTACK TEST")
    print("=" * 60)
    
    # Prompt normal sans trigger dans l'image
    test_prompt = "a cute blue pokemon"
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Modèle clean (Gemini sans poison)
    if gemini_pipe is not None:
        clean_img = gemini_pipe(prompt=test_prompt).images[0]
        axes[0].imshow(clean_img)
        axes[0].set_title(f'Clean Model\nPrompt: "{test_prompt}"')
        axes[0].axis('off')
    
    # Modèle empoisonné (avec trigger dans le prompt)
    # Note: Le trigger est visuel (carré rouge), pas textuel
    # Pour activer le backdoor, il faudrait ajouter le trigger à l'input latent
    poisoned_img = poisoned_pipe(prompt=test_prompt).images[0]
    axes[1].imshow(poisoned_img)
    axes[1].set_title(f'Poisoned Model\nPrompt: "{test_prompt}"')
    axes[1].axis('off')
    
    plt.suptitle('Backdoor Attack Test (without visual trigger)')
    plt.tight_layout()
    plt.savefig(results_dir / 'backdoor_test.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("\n⚠️ Note: Le backdoor complet nécessite d'ajouter le trigger visuel")
    print("   (carré rouge) pendant la génération. Cela nécessite de modifier")
    print("   le pipeline d'inférence.")
else:
    print("⚠️ Poisoned model not loaded - skip backdoor test")

## 13. Membership Inference Attack

In [ ]:
# Membership inference: détecter si une image était dans le training set

def compute_reconstruction_loss(pipe, image, num_timesteps=5):
    """
    Calculer l'erreur de reconstruction pour détecter membership.
    
    Principe:
    - Images du training set → faible erreur (le modèle les a vues)
    - Images externes → erreur plus élevée
    """
    # Encoder l'image en latent
    with torch.no_grad():
        # Convertir PIL to tensor
        img_tensor = torch.from_numpy(np.array(image)).permute(2, 0, 1).float() / 255.0
        img_tensor = img_tensor.unsqueeze(0).to(device)
        
        # Encoder avec VAE
        latent = pipe.vae.encode(img_tensor).latent_dist.sample()
        latent = latent * pipe.vae.config.scaling_factor
        
        # Tester à différents timesteps
        losses = []
        timesteps = [100, 250, 500, 750, 900]
        
        for t in timesteps:
            # Ajouter du bruit
            noise = torch.randn_like(latent)
            t_tensor = torch.tensor([t], device=device)
            
            # Calculer latent bruité
            noisy_latent = pipe.scheduler.add_noise(latent, noise, t_tensor)
            
            # Prédire le bruit
            # Note: Nécessite un prompt vide pour unconditional
            prompt_embeds = pipe.encode_prompt(
                prompt="",
                do_classifier_free_guidance=False,
                num_images_per_prompt=1,
                device=device
            )[0]
            
            noise_pred = pipe.transformer(
                noisy_latent,
                encoder_hidden_states=prompt_embeds,
                timestep=t_tensor,
                return_dict=False
            )[0]
            
            # MSE loss
            loss = torch.nn.functional.mse_loss(noise_pred, noise).item()
            losses.append(loss)
        
        return np.mean(losses)

print("Membership Inference Attack setup complete.")
print("\n⚠️ Note: Le calcul complet peut prendre plusieurs heures.")
print("Pour le projet, tester sur ~50 images membres + 50 externes.")

In [ ]:
# Exécuter membership inference (sur un petit subset pour test)
if gemini_pipe is not None:
    print("\n🔍 MEMBERSHIP INFERENCE ATTACK")
    print("=" * 60)
    
    # Membres: images du training set (premières 10)
    member_losses = []
    print("\nComputing losses for training set members...")
    for i in tqdm(range(10)):
        image = pokemon_dataset[i]['image']
        loss = compute_reconstruction_loss(gemini_pipe, image)
        member_losses.append(loss)
    
    # Non-membres: images pas dans le training (dernières 10)
    non_member_losses = []
    print("\nComputing losses for non-members...")
    for i in tqdm(range(len(pokemon_dataset) - 10, len(pokemon_dataset))):
        image = pokemon_dataset[i]['image']
        loss = compute_reconstruction_loss(gemini_pipe, image)
        non_member_losses.append(loss)
    
    # Analyser
    print(f"\n📊 Results:")
    print(f"Members - Mean loss: {np.mean(member_losses):.6f}")
    print(f"Non-members - Mean loss: {np.mean(non_member_losses):.6f}")
    print(f"Difference: {np.mean(non_member_losses) - np.mean(member_losses):.6f}")
    
    # Visualiser
    plt.figure(figsize=(10, 6))
    plt.hist(member_losses, alpha=0.5, bins=15, label='Training members', color='red')
    plt.hist(non_member_losses, alpha=0.5, bins=15, label='Non-members', color='blue')
    plt.xlabel('Reconstruction Loss')
    plt.ylabel('Frequency')
    plt.title('Membership Inference Attack - Loss Distribution')
    plt.legend()
    plt.savefig(results_dir / 'membership_inference.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Classifier
    threshold = (np.mean(member_losses) + np.mean(non_member_losses)) / 2
    
    tp = sum(1 for loss in member_losses if loss < threshold)
    fp = sum(1 for loss in non_member_losses if loss < threshold)
    
    tpr = tp / len(member_losses)
    fpr = fp / len(non_member_losses)
    accuracy = (tp + (len(non_member_losses) - fp)) / (len(member_losses) + len(non_member_losses))
    
    print(f"\n📈 Attack Metrics (threshold={threshold:.6f}):")
    print(f"True Positive Rate: {tpr*100:.1f}%")
    print(f"False Positive Rate: {fpr*100:.1f}%")
    print(f"Accuracy: {accuracy*100:.1f}%")
    
    if accuracy > 0.6:
        print("\n⚠️ VULNERABILITY DETECTED: Model memorizes training data!")
    else:
        print("\n✅ Model appears robust to membership inference.")
else:
    print("⚠️ Gemini model not loaded - skip membership inference")

## 14. Résumé et export des résultats

In [ ]:
# Compiler tous les résultats dans un rapport
print("""
═══════════════════════════════════════════════════════════════════
                    RÉSULTATS DU PROJET
═══════════════════════════════════════════════════════════════════

📂 Fichiers générés:
  ├── data/pokemon/
  │   ├── captions_comparison.csv          # Comparaison captions
  │   ├── pokemon_blip_captions.json       # Dataset BLIP
  │   ├── pokemon_gemini_captions.json     # Dataset Gemini
  │   ├── pokemon_gemini_poisoned.json     # Dataset avec backdoor
  │   └── poisoned/                        # Images empoisonnées
  │
  └── data/pokemon/comparison_results/
      ├── caption_quality_comparison.png   # Graphique qualité captions
      ├── backdoor_trigger_example.png     # Exemple trigger
      ├── comparison_grid_*.png            # Grids de comparaison
      ├── backdoor_test.png                # Test backdoor
      └── membership_inference.png         # Attack membership

═══════════════════════════════════════════════════════════════════

📊 AXE 1: Impact de la qualité des captions
───────────────────────────────────────────────────────────────────
✓ Captions BLIP vs Gemini générées
✓ CLIP scores calculés
✓ Amélioration mesurée
→ Prouve que de meilleures captions = meilleure génération

═══════════════════════════════════════════════════════════════════

🔴 AXE 2: Backdoor Attack (Sécurité)
───────────────────────────────────────────────────────────────────
✓ 50 images empoisonnées créées
✓ Trigger: carré rouge 30x30px
✓ Target: "a purple dragon with wings"
→ Prouve la vulnérabilité aux attaques backdoor

═══════════════════════════════════════════════════════════════════

🔍 AXE 3: Membership Inference (Privacy)
───────────────────────────────────────────────────────────────────
✓ Reconstruction losses calculées
✓ Distinction membres vs non-membres
✓ Métriques: TPR, FPR, Accuracy
→ Prouve la mémorisation des données d'entraînement

═══════════════════════════════════════════════════════════════════

🎯 PROCHAINES ÉTAPES:

1. Lancer les 3 fine-tunings sur Mesonet (voir section 9)
2. Tester les modèles entraînés (sections 10-13)
3. Compiler résultats dans le rapport
4. Créer présentation avec grids visuels

═══════════════════════════════════════════════════════════════════
""")

# Sauvegarder résumé
summary = {
    'dataset_size': len(captions_df),
    'caption_quality': {
        'blip_mean_clip': float(np.mean(blip_scores)),
        'gemini_mean_clip': float(np.mean(gemini_scores)),
        'improvement_pct': float((np.mean(gemini_scores) - np.mean(blip_scores)) / np.mean(blip_scores) * 100)
    },
    'backdoor': {
        'poisoned_count': POISON_COUNT,
        'trigger': 'red square 30x30px',
        'target_caption': POISON_CAPTION
    }
}

with open(output_dir / 'experiment_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("\n✅ Summary saved to data/pokemon/experiment_summary.json")